# Monitor web en Google Colab

## Cómo trabaja

El proceso usa una carpeta de trabajo persistente en Google Drive:

- `/content/drive/MyDrive/Proyectos/web-monitor-app`

Dentro de esa carpeta quedan los tres archivos de estado del monitoreo:

- `urls.txt`: lista de URLs a monitorear.
- `web_monitoring_hashes.json`: hashes de la última versión conocida de cada URL.
- `last_run_results.json`: resultados completos de la última ejecución.

## Estrategia de monitoreo

- Primero intenta leer cada sitio con `requests` y `BeautifulSoup`.
- Usa estrategias especiales para Google Drive y Zonaprop.
- Deja 5 sitios fuera del monitoreo automático para revisión manual.
- Solo usa `Playwright + Chromium` si vos activás el fallback opcional.

## Estructura del notebook

- Cada bloque de código está precedido por una celda explicativa.
- Los comentarios dentro del código están escritos en español.
- Si `urls.txt` no existe todavía, el notebook crea una versión inicial.


## Celda 1: instalar dependencias básicas

Esta celda instala las librerías necesarias para ejecutar el monitoreo en Colab. Incluye `nest_asyncio` para evitar problemas con el loop asíncrono dentro del notebook.

In [1]:
!pip install -q requests beautifulsoup4 nest_asyncio

## Celda 2: instalar Playwright de forma opcional

Esta celda solo hace falta si querés activar el fallback con navegador para las URLs que no puedan resolverse con `requests`. Si no la ejecutás, el monitoreo funciona igual en modo liviano.

In [2]:
# Ejecutá esta celda solo si querés habilitar el fallback con Playwright.
# Instala Playwright y tambi?n las librer?as del sistema que Chromium necesita en Colab.
!pip install -q playwright
!playwright install --with-deps chromium


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 17.7 MB/s eta 0:00:00
Installing dependencies...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,931 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jamm

## Celda 3: montar Google Drive y definir la carpeta de trabajo

Esta celda monta Google Drive y fija la carpeta persistente donde van a vivir la lista de URLs, los hashes y los resultados.

In [3]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Esta es la carpeta persistente del proyecto dentro de Google Drive.
BASE_DIR = Path('/content/drive/MyDrive/Proyectos/web-monitor-app')
BASE_DIR.mkdir(parents=True, exist_ok=True)

URLS_FILE = BASE_DIR / 'urls.txt'
HASHES_FILE = BASE_DIR / 'web_monitoring_hashes.json'
RESULTS_FILE = BASE_DIR / 'last_run_results.json'

print('Carpeta base:', BASE_DIR)
print('Archivo de URLs:', URLS_FILE)
print('Archivo de hashes:', HASHES_FILE)
print('Archivo de resultados:', RESULTS_FILE)

Mounted at /content/drive
Carpeta base: /content/drive/MyDrive/Proyectos/web-monitor-app
Archivo de URLs: /content/drive/MyDrive/Proyectos/web-monitor-app/urls.txt
Archivo de hashes: /content/drive/MyDrive/Proyectos/web-monitor-app/web_monitoring_hashes.json
Archivo de resultados: /content/drive/MyDrive/Proyectos/web-monitor-app/last_run_results.json


## Celda 4: definir toda la lógica del monitoreo

Esta es la celda principal del notebook. Acá se definen las funciones para leer URLs, calcular hashes, aplicar estrategias especiales, guardar resultados y ejecutar el monitoreo completo.

In [4]:
import asyncio
import hashlib
import json
from datetime import datetime

import nest_asyncio
import requests
from bs4 import BeautifulSoup

# En notebooks de Colab ya suele existir un loop activo.
# Esta línea permite usar asyncio sin errores de reentrada.
nest_asyncio.apply()

try:
    from playwright.async_api import async_playwright
except Exception:
    async_playwright = None


DEFAULT_USER_AGENT = (
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
)

REQUEST_HEADERS = {
    'User-Agent': DEFAULT_USER_AGENT,
    'Accept-Language': 'es-419,es;q=0.9',
}

PLAYWRIGHT_LAUNCH_ARGS = [
    '--disable-blink-features=AutomationControlled',
    '--disable-dev-shm-usage',
    '--no-sandbox',
]

REQUEST_TIMEOUT = 30
PAGE_GOTO_TIMEOUT_MS = 45000
PAGE_IDLE_TIMEOUT_MS = 3000
PAGE_TEXT_TIMEOUT_MS = 5000
REQUEST_TEXT_MIN_CHARS = 80

# Estos sitios dieron falsos positivos frecuentes y se revisan a mano.
MANUAL_REVIEW_URLS = {
    'https://adrianmercadorealestate.com/blog/informes',
    'https://www.fabianachaval.com/blog',
    'https://www.ljramos.com.ar/informes-del-mercado-inmobiliario',
    'https://www.cbre.com.ar/insights#market-reports',
}

# En estos dominios ya se comprobó que requests alcanza.
STATIC_REQUEST_DOMAINS = ('afcp.org.ar', 'uade.edu.ar')

ZONAPROP_PDF_PATTERNS = {
    'zpindex/gba-oeste-sur-venta': 'INDEX_GBA_OESTE_REPORTE_{year}-{month:02d}.pdf',
    'zpindex/gba-venta': 'INDEX_GBA_NORTE_REPORTE_{year}-{month:02d}.pdf',
    'zpindex/informe-demanda': 'INDEX_AMBA_REPORTE_DEMANDA-{year}-{month:02d}-PDF.pdf',
    'zpindex': 'INDEX_CABA_REPORTE_{year}-{month:02d}.pdf',
}


def get_requests_session():
    session = requests.Session()
    session.headers.update(REQUEST_HEADERS)
    return session


def load_urls(urls_file):
    return [
        line.strip()
        for line in urls_file.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]


def auto_monitor_urls(urls):
    return [url for url in urls if url not in MANUAL_REVIEW_URLS]


def is_gdrive_folder(url):
    return 'drive.google.com/drive/folders/' in url


def clean_text(text):
    return ' '.join(text.split())


def calculate_hash(text):
    if not text:
        return None
    return hashlib.sha256(text.encode('utf-8', errors='ignore')).hexdigest()


def extract_visible_text(html_content):
    if not html_content:
        return ''
    soup = BeautifulSoup(html_content, 'html.parser')
    return soup.get_text(separator=' ', strip=True)


def load_hashes():
    if HASHES_FILE.exists():
        return json.loads(HASHES_FILE.read_text(encoding='utf-8'))
    return {}


def save_hashes(hashes):
    HASHES_FILE.write_text(
        json.dumps(hashes, indent=2, ensure_ascii=False),
        encoding='utf-8',
    )


def save_results(results):
    payload = {
        'generated_at': datetime.now().isoformat(timespec='seconds'),
        'results': results,
    }
    RESULTS_FILE.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False),
        encoding='utf-8',
    )


def get_zonaprop_pdf_pattern(url):
    for key, pattern in ZONAPROP_PDF_PATTERNS.items():
        if key in url:
            return pattern
    return None


def fetch_zonaprop_pdf_from_page(url, session):
    import urllib3
    from urllib.parse import urljoin

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT, verify=False)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, 'html.parser')
        pdf_links = []
        for link in soup.find_all('a', href=True):
            href = link['href'].strip()
            text = link.get_text(' ', strip=True).lower()
            if '.pdf' not in href.lower():
                continue
            absolute_url = urljoin(url, href)
            score = 0
            if 'descarg' in text or 'informe' in text:
                score += 2
            if 'index' in absolute_url.lower() or 'reporte' in absolute_url.lower():
                score += 1
            pdf_links.append((score, absolute_url))

        if not pdf_links:
            return None

        pdf_links.sort(reverse=True)
        return pdf_links[0][1]
    except Exception:
        return None


def find_latest_zonaprop_pdf(pdf_pattern, session):
    import urllib3

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    now = datetime.now()
    base_url = 'https://www.zonaprop.com.ar/blog/wp-content/uploads'
    candidates = []

    # Primero buscamos por mes del informe. Zonaprop suele subir el PDF
    # en el mismo mes del informe o en los dos meses siguientes.
    for report_delta in range(18):
        report_month = now.month - report_delta
        report_year = now.year
        while report_month < 1:
            report_month += 12
            report_year -= 1

        for upload_offset in range(3):
            upload_month = report_month + upload_offset
            upload_year = report_year
            while upload_month > 12:
                upload_month -= 12
                upload_year += 1
            candidates.append((upload_year, upload_month, report_year, report_month))

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        if candidate not in seen:
            seen.add(candidate)
            unique_candidates.append(candidate)

    for upload_year, upload_month, report_year, report_month in unique_candidates:
        filename = pdf_pattern.format(year=report_year, month=report_month)
        pdf_url = f'{base_url}/{upload_year}/{upload_month:02d}/{filename}'
        try:
            response = session.head(pdf_url, timeout=10, verify=False, allow_redirects=True)
            if response.status_code == 200:
                return pdf_url
        except Exception:
            continue

    return None


def parse_gdrive_date(date_text):
    month_map = {
        'ene': 1,
        'feb': 2,
        'mar': 3,
        'abr': 4,
        'may': 5,
        'jun': 6,
        'jul': 7,
        'ago': 8,
        'sept': 9,
        'oct': 10,
        'nov': 11,
        'dic': 12,
    }
    cleaned = date_text.strip().lower().replace('.', '')
    parts = cleaned.split()
    if len(parts) != 3:
        return None
    day, month_text, year = parts
    month = month_map.get(month_text)
    if month is None:
        return None
    try:
        return datetime(int(year), month, int(day))
    except ValueError:
        return None


def fetch_gdrive_folder(url, session):
    import urllib3

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT, verify=False)
        if response.status_code != 200:
            return None, f'HTTP {response.status_code}'

        soup = BeautifulSoup(response.text, 'html.parser')
        entries = []
        for row in soup.select('tr[role="row"]'):
            name_el = row.select_one('td[data-column-field="6"] [data-tooltip]')
            date_el = row.select_one('td[data-column-field="5"] span')
            if not name_el or not date_el:
                continue

            file_name = name_el.get('data-tooltip', '').strip()
            file_name = file_name.removesuffix(' PDF').removesuffix(' PPTX').removesuffix(' PPT')
            modified_text = date_el.get_text(' ', strip=True)
            modified_dt = parse_gdrive_date(modified_text)
            if not file_name or modified_dt is None:
                continue

            entries.append({
                'name': file_name,
                'modified_text': modified_text,
                'modified_iso': modified_dt.date().isoformat(),
            })

        if not entries:
            return None, 'No se encontraron archivos con metadatos de fecha en la carpeta'

        deduped_entries = {(e['name'], e['modified_iso']): e for e in entries}
        sorted_entries = sorted(
            deduped_entries.values(),
            key=lambda entry: (entry['modified_iso'], entry['name']),
            reverse=True,
        )
        recent_entries = sorted_entries[:6]
        content = '\n'.join(
            f"{entry['modified_iso']}|{entry['modified_text']}|{entry['name']}"
            for entry in recent_entries
        )
        return content, ''
    except Exception as exc:
        return None, str(exc)


async def fetch_text_with_requests(session, url):
    import urllib3

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    loop = asyncio.get_running_loop()
    response = await loop.run_in_executor(
        None,
        lambda: session.get(url, timeout=REQUEST_TIMEOUT, verify=False),
    )
    if response.status_code != 200:
        return None, f'HTTP Error {response.status_code}'

    content = extract_visible_text(response.text)
    if len(content) < REQUEST_TEXT_MIN_CHARS:
        return None, 'Contenido insuficiente (Requests)'
    return content, ''


async def fetch_with_playwright(browser, url):
    if browser is None:
        return url, 'Error', '', 'Browser no disponible'

    content = ''
    error_msg = ''
    status = 'Error'

    for _ in range(2):
        context = None
        page = None
        try:
            context = await browser.new_context(
                user_agent=DEFAULT_USER_AGENT,
                viewport={'width': 1920, 'height': 1080},
                device_scale_factor=1,
                extra_http_headers={
                    'Accept-Language': 'es-419,es;q=0.9',
                    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
                    'Referer': 'https://www.google.com/',
                },
            )
            page = await context.new_page()
            page.set_default_navigation_timeout(PAGE_GOTO_TIMEOUT_MS)
            page.set_default_timeout(PAGE_TEXT_TIMEOUT_MS)

            try:
                await page.goto(url, wait_until='domcontentloaded')
            except Exception:
                pass

            try:
                await page.wait_for_load_state('networkidle', timeout=PAGE_IDLE_TIMEOUT_MS)
            except Exception:
                pass

            content = await page.inner_text('body')
            if content:
                return url, 'Success', content, ''
            raise Exception('Empty content retrieved')
        except Exception as exc:
            error_msg = str(exc)
            status = 'Error'
        finally:
            if page:
                try:
                    await page.close()
                except Exception:
                    pass
            if context:
                try:
                    await context.close()
                except Exception:
                    pass

    return url, status, content, error_msg


async def fetch_url(session, url, use_playwright_fallback=False):
    if 'zonaprop.com.ar' in url:
        pdf_pattern = get_zonaprop_pdf_pattern(url)
        if pdf_pattern is None:
            return url, 'Error', '', 'URL de Zonaprop no reconocida'
        loop = asyncio.get_running_loop()
        pdf_url = await loop.run_in_executor(None, fetch_zonaprop_pdf_from_page, url, session)
        if not pdf_url:
            pdf_url = await loop.run_in_executor(None, find_latest_zonaprop_pdf, pdf_pattern, session)
        if pdf_url:
            return url, 'Success', f'PDF_URL: {pdf_url}', ''
        return url, 'Error', '', 'No se encontr? PDF reciente'

    if is_gdrive_folder(url):
        content, error = fetch_gdrive_folder(url, session)
        if content:
            return url, 'Success', content, ''
        return url, 'Error', '', error

    requests_error = ''
    try:
        content, requests_error = await fetch_text_with_requests(session, url)
        if content:
            return url, 'Success', content, ''
    except Exception as exc:
        requests_error = f'Requests Error: {exc}'

    if not use_playwright_fallback:
        return url, 'Error', '', requests_error or 'Requests no pudo leer la URL'

    if async_playwright is None:
        return url, 'Error', '', 'Playwright no está instalado'

    try:
        async with async_playwright() as playwright:
            browser = await playwright.chromium.launch(headless=True, args=PLAYWRIGHT_LAUNCH_ARGS)
            try:
                result = await fetch_with_playwright(browser, url)
            finally:
                await browser.close()
    except Exception as exc:
        if requests_error:
            return url, 'Error', '', f'{requests_error} | Playwright no pudo iniciar: {exc}'
        return url, 'Error', '', f'Playwright no pudo iniciar: {exc}'

    if result[1] == 'Success':
        return result

    if requests_error:
        return url, 'Error', '', f'{requests_error} | Playwright: {result[3]}'
    return result


async def process_urls(urls, use_playwright_fallback=False):
    session = get_requests_session()
    results = []
    for index, url in enumerate(urls, start=1):
        print(f'[{index}/{len(urls)}] {url}')
        result = await fetch_url(session, url, use_playwright_fallback=use_playwright_fallback)
        print(f'  -> {result[1]}')
        results.append(result)
    return results


def run_monitor(use_playwright_fallback=False):
    urls = load_urls(URLS_FILE)
    monitor_urls = auto_monitor_urls(urls)
    previous_hashes = load_hashes()
    new_hashes = previous_hashes.copy()

    raw_results = asyncio.run(process_urls(monitor_urls, use_playwright_fallback=use_playwright_fallback))
    processed_data = []
    changes_count = 0
    errors_count = 0

    for url, status, text, error_msg in raw_results:
        row = {
            'URL': url,
            'Estado': status,
            'Resultado': 'Sin cambios',
            'Error': error_msg if status == 'Error' else '',
        }

        if status == 'Success':
            current_hash = calculate_hash(clean_text(text))
            last_hash = previous_hashes.get(url)
            if last_hash is None:
                row['Resultado'] = 'Nuevo (primer rastreo)'
                new_hashes[url] = current_hash
            elif current_hash != last_hash:
                row['Resultado'] = 'Cambio detectado'
                new_hashes[url] = current_hash
                changes_count += 1
            else:
                row['Resultado'] = 'Sin cambios'
        else:
            row['Resultado'] = 'Error de lectura'
            errors_count += 1

        processed_data.append(row)

    save_hashes(new_hashes)
    save_results(processed_data)

    return {
        'generated_at': datetime.now().isoformat(timespec='seconds'),
        'processed_data': processed_data,
        'changes_count': changes_count,
        'errors_count': errors_count,
        'manual_review_urls': sorted(MANUAL_REVIEW_URLS),
    }


## Celda 5: configurar la corrida

En esta celda defin?s si quer?s usar el fallback opcional con Playwright. Lo recomendado es dejarlo en `False`. Solo conviene activarlo si detect?s una URL puntual que no puede resolverse con `requests` y ya ejecutaste la celda opcional de instalaci?n con dependencias.


In [5]:
# Cambiá este valor a True solo si antes ejecutaste la celda opcional de Playwright.
USAR_PLAYWRIGHT_FALLBACK = True

print('Fallback con Playwright activado:', USAR_PLAYWRIGHT_FALLBACK)
print('Carpeta de trabajo:', BASE_DIR)


Fallback con Playwright activado: True
Carpeta de trabajo: /content/drive/MyDrive/Proyectos/web-monitor-app


## Celda 6: ejecutar el monitoreo y mostrar el resumen

Esta celda corre el monitoreo completo, actualiza los hashes y guarda el archivo `last_run_results.json`. Adem?s, muestra dentro del propio notebook todas las URLs con cambios detectados, nuevos rastreos y errores de lectura.


In [6]:
summary = run_monitor(use_playwright_fallback=USAR_PLAYWRIGHT_FALLBACK)

cambios = [row for row in summary['processed_data'] if row['Resultado'] == 'Cambio detectado']
nuevos = [row for row in summary['processed_data'] if row['Resultado'] == 'Nuevo (primer rastreo)']
errores = [row for row in summary['processed_data'] if row['Resultado'] == 'Error de lectura']

print('\nResumen de la corrida')
print('Fecha:', summary['generated_at'])
print('URLs autom?ticas:', len(summary['processed_data']))
print('Cambios detectados:', len(cambios))
print('Nuevos primeros rastreos:', len(nuevos))
print('Errores:', len(errores))
print('\nSitios para revisi?n manual:')
for url in summary['manual_review_urls']:
    print('-', url)

print('\nURLs con cambios detectados:')
if cambios:
    for row in cambios:
        print('-', row['URL'])
else:
    print('No se detectaron cambios en esta corrida.')

print('\nURLs nuevas en primer rastreo:')
if nuevos:
    for row in nuevos:
        print('-', row['URL'])
else:
    print('No hubo URLs nuevas en esta corrida.')

print('\nURLs con error de lectura:')
if errores:
    for row in errores:
        print('-', row['URL'])
        if row['Error']:
            print('  Error:', row['Error'])
else:
    print('No hubo errores de lectura.')

print('\nArchivo de resultados guardado en:', RESULTS_FILE)
print('Archivo de hashes guardado en:', HASHES_FILE)


[1/26] https://buenosaires.gob.ar/jefaturadegabinete/desarrollo-urbano/normativa/codigo-urbanistico-y-de-edificacion
  -> Success
[2/26] https://buenosaires.gob.ar/jefaturadegabinete/desarrollo-urbano/novedades-de-la-subsecretaria-de-gestion-urbana
  -> Success
[3/26] https://buenosaires.gob.ar/jefaturadegabinete/desarrollo-urbano
  -> Success
[4/26] https://buenosaires.gob.ar/agc/noticias
  -> Success
[5/26] https://www.cpau.org/noticias
  -> Success
[6/26] https://buenosaires.gob.ar/noticias/desarrollo-urbano
  -> Success
[7/26] https://www.estadisticaciudad.gob.ar/eyc/?page_id=1479
  -> Success
[8/26] https://ceso.com.ar/producciones-publicaciones/
  -> Success
[9/26] https://colegioinmobiliario.org.ar/institucional/observatorioEstadistico
  -> Success
[10/26] https://colegioinmobiliario.org.ar/novedades
  -> Success
[11/26] https://www.afcp.org.ar/despacho-mensual
  -> Success
[12/26] https://www.indec.gob.ar/indec/web/Nivel3-Tema-3-3
  -> Success
[13/26] https://www.remax.com.ar/i